# Phase 2 · Chicago food inspection prediction

Predict Fail using inspection context and strictly earlier license-level history. Labels are Fail (1), Pass and Pass w/ Conditions (0). Other outcomes can contribute prior visits but are excluded from classification.

Run from the repository root. The tracked October 12 snapshot is verified against its stored checksum. The experiment processes labels through June 30, 2026. July 1–October 9 is reserved for final model evaluation. Phase 1 already described some 2026 outcomes, so the local holdout was previously seen in aggregate. The official hidden period starts October 31.

## Design

- M0 uses Facility Type, Risk and Inspection Type. M1 adds prior visits, fail and judged counts, previous-day outcome, the interaction with the current inspection type, a smoothed fail rate and a history flag. M2 adds days since the previous visit and previous fail.
- Same-day rows share the same prior history. Discordant results on the previous visit date receive a mixed category. Unknown licenses have no matched history. The pooled smoothing prior uses only earlier days in the permitted source.
- Logistic regression and histogram gradient boosting use preprocessing fitted in each training window. The 2019 start changes model fitting only and still allows pre-2019 visits in establishment history.
- Each validation window freezes outcome history at its start, matching batch scoring. The 2023 fold uses a threshold of 0.50. Subsequent folds select a threshold in 0.10–0.60 from pooled predictions on earlier folds. Rank configurations by mean 2024–2025 Macro-F1, breaking ties by the lower yearly score. Compare the two leaders and the full-window M0 logistic baseline on January–June 2026 at thresholds fixed from the historical folds.

In [1]:
from temporal_modeling import run

historical, ranking, validation = run()

In [2]:
historical.round(4)

,year,algorithm,features,start_year,threshold,n,fail_rate,macro_f1,pass_precision,pass_recall,pass_f1,fail_precision,fail_recall,fail_f1,tn,fp,fn,tp
0,2023,logistic,M0,2016,0.50,15413,0.2488,0.4296,0.7512,0.9994,0.8577,0.3000,0.0008,0.0016,11571,7,3832,3
1,2023,logistic,M0,2019,0.50,15413,0.2488,0.4291,0.7511,0.9995,0.8577,0.1429,0.0003,0.0005,11572,6,3834,1
2,2023,logistic,M1,2016,0.50,15413,0.2488,0.4551,0.7549,0.9917,0.8572,0.5271,0.0279,0.0530,11482,96,3728,107
3,2023,logistic,M1,2019,0.50,15413,0.2488,0.4511,0.7544,0.9938,0.8577,0.5528,0.0232,0.0445,11506,72,3746,89
4,2023,logistic,M2,2016,0.50,15413,0.2488,0.4683,0.7571,0.9881,0.8573,0.5430,0.0428,0.0793,11440,138,3671,164
5,2023,logistic,M2,2019,0.50,15413,0.2488,0.4663,0.7568,0.9889,0.8574,0.5458,0.0404,0.0753,11449,129,3680,155
6,2023,boosting,M0,2016,0.50,15413,0.2488,0.4322,0.7516,0.9986,0.8577,0.4483,0.0034,0.0067,11562,16,3822,13
7,2023,boosting,M0,2019,0.50,15413,0.2488,0.4326,0.7515,0.9980,0.8574,0.3947,0.0039,0.0077,11555,23,3820,15
8,2023,boosting,M1,2016,0.50,15413,0.2488,0.4539,0.7547,0.9918,0.8571,0.5178,0.0266,0.0506,11483,95,3733,102
9,2023,boosting,M1,2019,0.50,15413,0.2488,0.4601,0.7557,0.9905,0.8573,0.5378,0.0334,0.0629,11468,110,3707,128


In [3]:
ranking.round(4)

mean     min
algorithm features start_year                
boosting  M1       2016        0.5830  0.5797
                   2019        0.5820  0.5790
          M2       2016        0.5804  0.5770
logistic  M2       2016        0.5788  0.5776
boosting  M2       2019        0.5786  0.5721
logistic  M1       2019        0.5762  0.5761
                   2016        0.5743  0.5739
          M2       2019        0.5741  0.5724
boosting  M0       2016        0.5570  0.5560
logistic  M0       2019        0.5549  0.5527
boosting  M0       2019        0.5547  0.5540
logistic  M0       2016        0.5124  0.4733

In [4]:
validation.round(4)

,algorithm,features,start_year,threshold,group,n,fail_rate,macro_f1,pass_precision,pass_recall,pass_f1,fail_precision,fail_recall,fail_f1,tn,fp,fn,tp
0,logistic,M0,2016,0.3,all,7557,0.2038,0.5543,0.8158,0.8664,0.8403,0.3111,0.2357,0.2682,5213,804,1177,363
1,logistic,M0,2016,0.3,with_history,6561,0.2027,0.5546,0.8166,0.8744,0.8445,0.3156,0.2278,0.2646,4574,657,1027,303
2,logistic,M0,2016,0.3,cold_start,996,0.2108,0.5496,0.8099,0.8130,0.8114,0.2899,0.2857,0.2878,639,147,150,60
3,boosting,M1,2016,0.3,all,7557,0.2038,0.5835,0.8324,0.8124,0.8223,0.3300,0.3610,0.3448,4888,1129,984,556
4,boosting,M1,2016,0.3,with_history,6561,0.2027,0.5880,0.8373,0.8008,0.8186,0.3312,0.3880,0.3573,4189,1042,814,516
5,boosting,M1,2016,0.3,cold_start,996,0.2108,0.5411,0.8044,0.8893,0.8447,0.3150,0.1905,0.2374,699,87,170,40
6,boosting,M1,2019,0.3,all,7557,0.2038,0.5814,0.8320,0.8075,0.8196,0.3256,0.3630,0.3433,4859,1158,981,559
7,boosting,M1,2019,0.3,with_history,6561,0.2027,0.5865,0.8376,0.7937,0.8151,0.3273,0.3947,0.3579,4152,1079,805,525
8,boosting,M1,2019,0.3,cold_start,996,0.2108,0.5289,0.8007,0.8995,0.8472,0.3009,0.1619,0.2105,707,79,176,34


## Validation findings

- The 2016-start boosting M1 led the historical ranking with mean Macro-F1 0.5830 across 2024–2025. The matching M0 scored 0.5570 and M2 scored 0.5804. The historical signal improved performance within the same boosting model. The two extra interval features did not improve the mean in this comparison.
- On January–June 2026 (7,557 labeled inspections), the 2016-start boosting M1 scored 0.5835 at a threshold of 0.30, versus 0.5543 for the logistic M0 baseline and 0.5814 for boosting M1 starting in 2019.
- Compared with logistic M0, the leading candidate found 193 more failures and flagged 325 additional non-failures. For records without matched history, Macro-F1 was 0.5411 versus 0.5496 for the baseline, with 996 inspections in this subgroup.
- The full-window boosting M1 at 0.30 is the leading candidate for the reserved local evaluation. Historic and 2026 validation scores are model-selection evidence. The final test and official test predictions remain pending.